# 03 · Embeddings: from text to vectors and 3D maps

An embedding is a vector of numbers learned for a specific model task. Here, we embed sections of six Acme documents in **the same text space**. The vector does not contain simple fields like "amount" or "date"; its coordinates are not human-readable labels. Similarity is useful for finding candidates, but it does not signify truth, authorization, or the correct answer.

In [ ]:
# Only cell with imports and project path.
import base64
import os
import sys
from io import BytesIO
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import numpy as np
import umap
from dotenv import dotenv_values
from PIL import Image as PILImage, ImageDraw
from sklearn.cluster import KMeans

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import DEMO_IDS, embed_texts, ollama_settings, section_records

## 1. What exactly are we embedding?

We do not pass access metadata to the model as part of the semantic text. We keep it alongside the vector. Each record indicates the version and hash of a specific section; a change in content requires a re-embedding.

In [ ]:
records = section_records()
host, model_name = ollama_settings()
vectors = embed_texts([row["text"] for row in records])
{"documents": len(DEMO_IDS), "sections": len(records), "shape": vectors.shape, "model": model_name,
 "first_record": {key: records[0][key] for key in ("id", "heading", "content_hash")}}

## 2. Direction and length of the vector

Cosine compares directions: `dot(a,b) / (||a||·||b||)`. Do not compare vectors from different models or dimensions. Normalizing to a length of 1 allows the use of the dot product, but it does not fix incorrect or truncated chunks.

In [ ]:
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
finance = next(i for i, row in enumerate(records) if row["document_id"] == "FIN-YTD-2011" and "Monthly" in row["heading"])
forecast = next(i for i, row in enumerate(records) if row["document_id"] == "SCI-FORECAST-2011" and "Error" in row["heading"])
{"finance_heading": records[finance]["heading"], "forecast_heading": records[forecast]["heading"],
 "cosine": float(unit[finance] @ unit[forecast]), "unit_norm": float(np.linalg.norm(unit[finance]))}

## 3. UMAP 3D and Clustering

First, we cluster the **original vectors** (experiment: two sets), then we plot their UMAP projection. The document color represents the known source label, while the cluster color represents the algorithm's result. UMAP distorts distances and density; the position in 3D is not a "true map of meaning," and KMeans forces a specific number of groups. Use the plot for diagnosis, not to declare the quality of the retrieval.

In [ ]:
clusters = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(unit)
coords = umap.UMAP(n_components=3, n_neighbors=min(10, len(records)-1), metric="cosine", random_state=42).fit_transform(unit)
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
colors = ["#2358a8" if row["document_id"].startswith("FIN") else "#dc7236" for row in records]
ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=colors, marker="o", alpha=.75)
ax.set(title="Acme Sections: blue = FIN, orange = SCI", xlabel="UMAP 1", ylabel="UMAP 2", zlabel="UMAP 3")
plt.show()
{"cluster_sizes": np.bincount(clusters).tolist(), "finance_in_cluster_0": sum(row["document_id"].startswith("FIN") and label == 0 for row, label in zip(records, clusters))}

## 4. Text, table, image: three different representations

- Text: embed a specific section or question with the same model.
- Table: embed the description with column headers, keeping numbers and the source row separate; for calculations, return to the structural data.
- Image: a text model can embed the **caption** from lesson 02, but not the pixels. A shared image-text space requires a different, specially trained multimodal model and a consistent `embed` function for both modalities.

Below, we build a small image without the source file from the previous lesson. `embeddinggemma-2` embeds pixels and both texts in **one own** 768D space. We do not compare this result with the 1024D Qwen vectors above.

In [ ]:
image = PILImage.new("RGB", (500, 110), "white")
ImageDraw.Draw(image).text((18, 45), "FORECAST  ->  ACTUAL  ->  ERROR", fill="black")
buffer = BytesIO()
image.save(buffer, format="PNG")
multimodal_model = os.environ.get("RAG_MULTIMODAL_EMBED_MODEL") or dotenv_values(ROOT / ".env").get("RAG_MULTIMODAL_EMBED_MODEL") or "embeddinggemma-2:latest"
mixed_input = [{"image": base64.b64encode(buffer.getvalue()).decode("ascii")},
               {"text": "A diagram comparing forecast, actual demand and error"},
               {"text": "A report on cash receipts and allocations"}]
response = httpx.post(host.rstrip("/") + "/api/embed", json={"model": multimodal_model, "input": mixed_input, "truncate": False}, timeout=180)
response.raise_for_status()
shared = np.asarray(response.json()["embeddings"], dtype=np.float32)
shared /= np.linalg.norm(shared, axis=1, keepdims=True)
{"shape": shared.shape, "image_to_forecast_text": float(shared[0] @ shared[1]),
 "image_to_cash_text": float(shared[0] @ shared[2])}

**Note:** two cosines for a single drawing do not measure model quality. You need a set of labeled images and questions to evaluate cross-modal retrieval.

**Exercise:** change `n_neighbors` and the number of clusters. Are the financial and scientific snippets still separated? Check this on the original texts, not just on the chart colors.